In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('../data/raw/media_performance_raw.csv')
print(df.groupby('channel')['conversions'].agg(['mean','median']))

                  mean  median
channel                       
Display       6.247956     6.0
Email        24.010944    22.0
Paid Search  88.791545    82.5
Paid Social  27.272069    25.0
PaidSearch   99.823529    90.0
Paid_Search  78.941176    77.0
Social       22.227273    21.0
Video         6.688797     6.0
paid search  81.941176    78.0
paid_social  30.047619    25.0


In [4]:
df['channel'].value_counts()

channel
Display        734
Email          731
Video          723
Paid Social    691
Paid Search    686
Social          22
paid_social     21
Paid_Search     17
paid search     17
PaidSearch      17
Name: count, dtype: int64

In [5]:
df[df['channel'] == 'Social']['spend'].describe()

count      22.000000
mean      820.267727
std       211.989291
min       524.340000
25%       649.605000
50%       774.630000
75%       965.250000
max      1247.460000
Name: spend, dtype: float64

### Hallazgo 01 — Etiquetas inconsistentes en `channel`

**Detección:** un `groupby('channel')` para comparar media/mediana reveló 10
etiquetas donde debían existir 5 canales.

**Diagnóstico:** `value_counts()` mostró un corte natural — 5 etiquetas con ~700
filas (canónicas) y 5 con 17-22 filas (variantes mal escritas: mayúsculas,
guiones, espacios).

**Caso ambiguo:** `Social` (22 filas) podía ser variante de `Paid Social` o un
canal orgánico legítimo. Se verificó con `spend.describe()`: min = 524 (ningún
cero) → es tráfico pagado → variante de `Paid Social`.

**Fix:** `.replace()` con diccionario de mapeo. Resultado: 5 canales canónicos.

In [9]:
mapa_canales = {
    'PaidSearch': 'Paid Search',
    'Paid_Search': 'Paid Search',
    'paid search': 'Paid Search',
    'Social': 'Paid Social',
    'paid_social': 'Paid Social'
}

df['channel'] = df['channel'].replace(mapa_canales)

In [10]:
print(df['channel'].value_counts())

channel
Paid Search    737
Paid Social    734
Display        734
Email          731
Video          723
Name: count, dtype: int64
